# Sensorredundantie op de productielijn verminderen met PROC GVARCLUS

## Samenvatting

Een productielijn met meerdere zones streamt tientallen gecorreleerde sensormetingen, waarvan er veel hetzelfde onderliggende signaal dragen. Deze notebook gebruikt **PROC GVARCLUS** (variabelenclustering) om de 13 processensoren te groeperen in vier disjuncte clusters, en leest vervolgens de R-kwadraatwaarden van elk cluster om per cluster één representatieve sensor te kiezen — wat de SPC-bewakingsvoetafdruk verkleint zonder procesinformatie te verliezen. Drie van de vier clusters komen netjes overeen met fysieke zones (gemiddelde R-kwadraat 0,92, 0,93 en 0,96); de vierde is een cluster met één kanaal dat de procedure heeft afgesplitst, wat we onderzoeken in plaats van te negeren.

## Gegevensbronnen

Alle gegevens worden inline gegenereerd met `call streaminit(20260531)` en `rand()` — geen externe of netwerkinvoer.

| Dataset | Rijen | Belangrijkste variabelen | Beschrijving |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | Synthetische metingen van een productielijn met 3 zones. Sensoren binnen een zone delen een latente driver (hoge correlatie); kruiszone-meters (temperaturen gekoppeld aan zone 1, drukken aan zone 2, trilling aan zone 3) zijn toegevoegd om realistische redundantie te creëren. De lus in de DATA-stap vraagt 300 cycli aan, maar deze evaluatieomgeving draait ongelicentieerd en behoudt de eerste 100 observaties — genoeg om de clusterstructuur netjes te herstellen. |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | Eén rij per invoersensor: het cluster waaraan hij is toegewezen en zijn R-kwadraat met de component van dat cluster. Geproduceerd door de `OUTPUT OUT=`-instructie. |

# Sensorredundantie op de productielijn verminderen met PROC GVARCLUS

Op een geïnstrumenteerde productielijn is het gebruikelijk om tientallen sensoren te loggen die overlappende fysieke verschijnselen meten — meerdere thermokoppels in één zone, redundante druktransducers, trillingssondes die dezelfde as volgen. Elk kanaal invoeren in een regelkaart of downstream model verspilt bewakingsinspanning en verhoogt de multicollineariteit.

**Variabelenclustering** pakt dit direct aan. `PROC GVARCLUS` verdeelt de numerieke variabelen in *disjuncte* clusters, waarbij elk cluster wordt samengevat door de eerste hoofdcomponent van zijn leden. Sensoren die samen bewegen komen in hetzelfde cluster terecht; de engineer kan dan één representatief kanaal per cluster behouden.

In deze notebook:

1. Genereren we metingen van een lijn met 3 zones met opzettelijk redundante sensoren (de lus vraagt 300 cycli aan; hier worden er 100 behouden).
2. Clusteren we de 13 kanalen met `PROC GVARCLUS` op `MAXCLUSTERS=4`.
3. Leggen we de clustertoewijzingen vast in een uitvoerdataset en vatten we ze samen.
4. Interpreteren we de R-kwadraatwaarden om per cluster één sensor te kiezen voor doorlopende SPC.

## Stap 1 — Synthetische sensorgegevens genereren

We simuleren drie proceszones, elk met een verborgen driver (`zone1_a`, `zone2_a`, `zone3_a`). De overige kanalen zijn ruizige lineaire functies van de driver van hun zone, waardoor ze sterk gecorreleerd zijn binnen een zone maar bijna onafhankelijk tussen zones. We koppelen ook de inlaat-/uitlaattemperaturen aan zone 1 en de twee druktransducers aan zone 2, wat de kruisinstrument-redundantie nabootst die op echte lijnen voorkomt. Een vaste seed maakt de run reproduceerbaar. De lus vraagt 300 cycli aan; in ongelicentieerde modus behoudt de engine de eerste 100 observaties, wat de onderstaande NOTE meldt.

In [1]:
GEGEVENS process_sensors;
    CALL streaminit(20260531);
    DOE cycle = 1 TOT 300;
        /* Zone 1: latente driver plus twee redundante sondes */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* Zone 2: latente driver plus twee redundante sondes */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* Zone 3: latente driver plus één redundante sonde */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* Kruisinstrument-kanalen gekoppeld aan bestaande drivers */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        UITVOER;
    EINDE;
    VERWIJDEREN cycle;
UITVOEREN;


NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.08 seconds
  cpu   0.08 seconds


## Stap 2 — De sensoren clusteren

We roepen `PROC GVARCLUS` aan op alle 13 kanalen. De `VAR`-instructie somt de numerieke sensoren op die geclusterd moeten worden. We begrenzen de partitie op vier clusters met `MAXCLUSTERS=4` (we verwachten ongeveer één cluster per fysieke zone, met wat speling). `ODS GRAPHICS ON` met `PLOTS=ALL` schakelt het variabelencluster-dendrogram in; de engine schrijft die SVG naar de werkmap in plaats van deze inline te renderen, dus de structuur die we hieronder lezen komt uit de afgedrukte tabel **Variable Cluster Assignments** en de eigenwaardetabel per cluster.

De `OUTPUT OUT=`-instructie schrijft de toewijzingen van variabele aan cluster — samen met de R-kwadraat van elke variabele ten opzichte van zijn eigen cluster — naar een dataset waarmee we later verder kunnen programmeren.

In [2]:
ODS GRAPHICS ON;

PROCEDURE gvarclus GEGEVENS=process_sensors maxclusters=4 PLOTS=ALL;
    VARIABELE zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    UITVOER out=clusters;
UITVOEREN;

ODS GRAPHICS OFF;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## Stap 3 — Opnieuw uitvoeren met de optie SUMMARY

De procedure een tweede keer uitvoeren met de optie `SUMMARY` behoudt dezelfde partitie. `PLOTS=NONE` schakelt grafieken uit voor deze doorloop. In deze omgeving is het afgedrukte rapport identiek aan Stap 2 — dezelfde tabel met 13 rijen toewijzingen, dezelfde vier clusters en dezelfde eigenwaardesamenvatting — dus deze cel toont vooral aan dat de opties `SUMMARY` en `PLOTS=NONE` correct worden geparsed en uitgevoerd; er worden geen nieuwe cijfers verwacht.

In [3]:
PROCEDURE gvarclus GEGEVENS=process_sensors maxclusters=4 summary PLOTS=none;
    VARIABELE zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
UITVOEREN;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## Stap 4 — De uitvoerdataset inspecteren

De dataset `clusters` uit `OUTPUT OUT=` heeft één rij per sensor met zijn toegewezen `Cluster` en `RSq_Own` (de kwadratische correlatie tussen de sensor en de component van zijn cluster). Een hoge `RSq_Own` betekent dat de sensor goed wordt vertegenwoordigd door zijn cluster — een ideale kandidaat om te *laten vallen* ten gunste van de representant van het cluster. We sorteren op cluster, en dan op R-kwadraat, zodat de meest representatieve sensor van elk cluster bovenaan zijn groep staat.

In [4]:
PROCEDURE SORTEREN GEGEVENS=clusters out=clusters_ranked;
    VOLGENS Cluster AFLOPEND RSq_Own;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=clusters_ranked label;
    VARIABELE Variable Cluster RSq_Own;
    label Variable = "Sensorkanaal"
          Cluster  = "Cluster"
          RSq_Own  = "R-kwadraat met eigen cluster";
UITVOEREN;


  Obs  Sensorkanaal  Cluster  R-kwadraat met eigen cluster
-----  ------------  -------  ----------------------------
    1  ZONE1_A             1                       0.96867
    2  ZONE1_B             1                        0.9189
    3  TEMP_IN             1                      0.903394
    4  TEMP_OUT            1                      0.889519
    5  ZONE2_A             2                       0.98364
    6  ZONE2_B             2                      0.946708
    7  PRESSURE_A          2                      0.929356
    8  PRESSURE_B          2                      0.920915
    9  ZONE2_C             2                      0.892405
   10  ZONE3_A             3                      0.977237
   11  VIBRATION           3                       0.95916
   12  ZONE3_B             3                      0.949054
   13  ZONE1_C             4                             1




NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## De resultaten interpreteren

De partitie herstelt het grootste deel van de fysieke structuur van de lijn, met één eerlijk voorbehoud:

- **Cluster 1** verzamelt `zone1_a` (R²=0,969), `zone1_b` (0,919) en de inlaat-/uitlaattemperaturen `temp_in` (0,903) en `temp_out` (0,890) — vier van de vijf kanalen aangestuurd door het latente signaal van zone 1. Gemiddelde R-kwadraat 0,920.
- **Cluster 2** verzamelt alle drie de zone-2-kanalen `zone2_a` (0,984), `zone2_b` (0,947), `zone2_c` (0,892) samen met de twee druktransducers `pressure_a` (0,929) en `pressure_b` (0,921), die we gekoppeld hebben aan de zone-2-driver. Gemiddelde R-kwadraat 0,935.
- **Cluster 3** verzamelt `zone3_a` (0,977), `zone3_b` (0,949) en `vibration` (0,959). Gemiddelde R-kwadraat 0,962.
- **Cluster 4** is een enkel kanaal: `zone1_c`, de derde zone-1-sonde, werd apart afgesplitst met R²=1,000 (een singleton wordt, triviaal, perfect door zichzelf verklaard). Ondanks dat het de zone-1-driver deelt, beoordeelde de procedure bij deze steekproefomvang `zone1_c` als voldoende onderscheidend van de `zone1_a`/temperatuurgroep om een eigen cluster te rechtvaardigen in plaats van samenvoeging met Cluster 1.

De drie multikanaals-clusters hebben elk een gemiddelde R-kwadraat boven **0,90**, wat bevestigt dat één component het grootste deel van de variatie tussen hun leden verklaart — precies de redundantie die we willen samenvoegen. De enkele uitschieter — `zone1_c` die in zijn eigen cluster terechtkomt in plaats van bij de rest van zone 1 — is een nuttige herinnering dat variabelenclustering datagedreven is: met meer cycli of een hoger `MAXCLUSTERS`-budget kan de grens verschuiven, dus de partitie is een startpunt voor technisch oordeel, geen vaste waarheid.

**Actie voor de lijn.** Behoud binnen elk multikanaals-cluster de sensor met de hoogste `RSq_Own` (het kanaal dat zijn cluster het best vertegenwoordigt) en trek de rest terug of geef ze lagere prioriteit in de routinematige SPC-registratie — bijvoorbeeld `zone2_a` voor Cluster 2 en `zone3_a` voor Cluster 3. Behandel de singleton `zone1_c` als een signaal om te onderzoeken in plaats van een automatische keuze om te behouden: bevestig of hij werkelijk onderscheidende informatie draagt of een clusterartefact is voordat je besluit hem apart te monitoren. Zelfs met dat ene kanaal apart gehouden, brengt dit 13 gemonitorde kanalen terug tot een monitoringplan met vier kanalen, wat alarmruis en multicollineariteit vermindert terwijl de informatie-inhoud van de lijn behouden blijft.